# 02 — Latent Class Model & Functional Class Validation

This template fits a **2-class Negative Binomial latent class model**
on Example 16-3 and then tests whether the discovered latent classes
align with the known **Functional Class (FC)** categories in the data.

**Hypothesis**: road sites belong to latent risk sub-populations that
are (at least partially) predicted by their functional classification
(freeway, principal arterial, minor arterial, collector, local road).

**Workflow**
1. Load data and prepare exposure offset
2. Declare constraints — FC drives class membership only
3. Run 2-class LC structure search
4. Extract class probabilities and assign each site to its most-likely class
5. Cross-tabulate predicted class vs actual FC label (chi-square test)
6. Load and re-fit the pre-specified book model

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

from metacountregressor import (
    ExperimentBuilder,
    ModelConstraints,
    load_example16_3_model_data,
    load_book_latent_class_spec,
    describe_book_latent_class_spec,
    get_help,
)

# Full latent-class usage guide:
# get_help('latent_class')

## 1 · Load data

In [ ]:
df = load_example16_3_model_data()

# Exposure offset
exposure = df['LENGTH'] * df['AADT'] * 365 / 1e8
df['OFFSET'] = np.log(exposure.clip(lower=1e-9))

# Functional class distribution
print('Functional class distribution:')
print(df['FC_LABEL'].value_counts().sort_index())
df[['ID', 'FREQ', 'FC_ENCODED', 'FC_LABEL', 'OFFSET']].head(5)

## 2 · Structural constraints

Key decision: `FC_ENCODED` enters **only** the class-membership equation
(role 7).  It does not directly affect the outcome — instead it shapes
the probability that a site belongs to the high-risk vs low-risk class.

In [ ]:
# Candidate outcome variables (excluding FC_ENCODED from outcome equation)
variables = [
    'URB', 'ACCESS', 'GRADEBR', 'CURVES', 'LENGTH',
    'SPEED', 'WIDTH', 'SLOPE', 'AVEPRE', 'FC_ENCODED',
]

c = (
    ModelConstraints()
    # FC_ENCODED drives class membership probability only
    .membership_only('FC_ENCODED')
    # Exposure must always be included
    .force_include('OFFSET')
    # Road geometry variables are not plausible ZI terms
    .no_zi('LENGTH', 'CURVES', 'GRADEBR', 'WIDTH', 'SLOPE')
    # Allow curvature random parameter
    .allow_random('CURVES', distributions=['lognormal'])
    # Binary dummies — no random effect needed
    .no_random('URB', 'GRADEBR')
)

print(c)

## 3 · Build experiment and run search

In [ ]:
builder = ExperimentBuilder(
    df=df,
    id_col='ID',
    y_col='FREQ',
    offset_col='OFFSET',
)

evaluator = builder.build_evaluator(
    variables=variables,
    constraints=c,
    max_latent_classes=2,
    # Include membership roles (7, 8) in the default search space
    default_roles=[0, 1, 2, 3, 5, 7, 8],
    mode='single',
    R=150,
)

In [ ]:
result = builder.run(
    evaluator=evaluator,
    algo='sa',
    max_iter=30,    # increase to 100+ for production runs
    seed=1,
)

print('Best BIC:', result.best_score)
print('Best structure:', result.best_solution)

## 4 · Extract class probabilities and assign classes

In [ ]:
# Posterior class probabilities — shape (N, C)
try:
    probs = builder.get_class_probabilities(result)
    df['pred_class'] = probs.argmax(axis=1) + 1    # 1-indexed
    df['class_prob_max'] = probs.max(axis=1)        # confidence
    print('Class assignment complete.')
    print(df['pred_class'].value_counts())
except AttributeError:
    # Fallback: if the method is not available, inspect result directly
    print('get_class_probabilities not available for this result type.')
    print('Check result.best_spec for the model structure.')
    print(result)

## 5 · Validate classes against actual functional class

In [ ]:
if 'pred_class' in df.columns:
    ct = pd.crosstab(
        df['FC_LABEL'],
        df['pred_class'],
        rownames=['Functional Class'],
        colnames=['Predicted Class'],
    )
    print('\nContingency table (counts):')
    print(ct)

    print('\nRow proportions (Pr[class | FC]):')
    print(ct.div(ct.sum(axis=1), axis=0).round(2))

    # Chi-square test of independence
    chi2, p, dof, expected = chi2_contingency(ct)
    print(f'\nChi-square test:  chi2={chi2:.2f},  p={p:.4f},  df={dof}')
    if p < 0.05:
        print('=> Latent classes are significantly associated with FC (p < 0.05)')
    else:
        print('=> No significant association between latent classes and FC')

## 6 · Pre-fitted book specification

The `load_book_latent_class_spec()` function returns the structural
specification (which variables, which roles, which distributions) of a
2-class NB latent class model representative of the Example 16-3
crash-frequency literature.  Pass it to `fit_manual_model()` to recover
the numerical estimates.

In [ ]:
# Describe the book specification
describe_book_latent_class_spec()

In [ ]:
# Re-fit the book model on Example 16-3 data
spec = load_book_latent_class_spec()

fit = builder.fit_manual_model(
    manual_spec=spec,
    model='nb',
    R=200,
)
print(fit)

## 7 · Interpretation guide

| Finding | Interpretation |
|---------|----------------|
| p < 0.05 in chi-square | FC is significantly associated with latent class — classes capture functional heterogeneity |
| One FC dominates a class | That functional class has a distinct crash-risk profile |
| FC_ENCODED BIC improves vs null | Including FC in membership improves model fit |
| Class 1 >> Class 2 in size | Check whether the minority class is statistically supported (compare BIC of 1 vs 2 classes) |

**Next steps**
- Compare 1-class vs 2-class BIC: `get_help('latent_class')`
- CMF model: `03_cmf_aadt_search.ipynb`
- Linear speed model: `04_linear_speed_prediction.ipynb`